In [ ]:
from typing import Optional, Dict, Any


class PromptBuilder:
    """Builds optimized prompts for product image generation."""

    def __init__(self):
        """Initialize the PromptBuilder."""
        pass

    def build_product_prompt(
        self,
        product_description: str,
        style: Optional[str] = None,
        composition: Optional[str] = None,
        lighting: Optional[str] = None
    ) -> str:
        """
        Build comprehensive prompt for product image generation.
        """
        style_desc = style or "photorealistic"
        composition_desc = composition or "centered product"
        lighting_desc = lighting or "soft studio lighting"
        quality_indicators = "high detail, professional photography, sharp focus, clean background"

        prompt = f"{style_desc} image of {product_description}, {composition_desc}, "
        prompt += f"{lighting_desc}, {quality_indicators}"

        return prompt


In [ ]:
import os
from pathlib import Path
from typing import Optional, Dict, Any
import requests
import uuid
from api_client import get_openai_client
from config import IMAGE_MODEL, DEFAULT_SIZE, DEFAULT_QUALITY, IMAGE_STORAGE_DIR


class ImageGenerator:
    """Generates product images using OpenAI Image API."""

    def __init__(self):
        """Initialize the ImageGenerator."""
        self.client = get_openai_client()
        os.makedirs(IMAGE_STORAGE_DIR, exist_ok=True)

    def generate_image(
        self,
        prompt: str,
        size: str = DEFAULT_SIZE,
        quality: str = DEFAULT_QUALITY
    ) -> Dict[str, Any]:
        """
        Generate image from prompt and download it.
        """
        try:
            response = self.client.images_generate(
                model=IMAGE_MODEL,
                prompt=prompt,
                size=size,
                quality=quality
            )

            image_url = response["data"][0]["url"]

            # Download image from URL
            image_filename = f"{uuid.uuid4()}.png"
            image_path = os.path.join(IMAGE_STORAGE_DIR, image_filename)
            downloaded_path = self.download_image(image_url, image_path)

            return {
                "image_path": downloaded_path,
                "url": image_url
            }

        except Exception as e:
            if hasattr(e, 'type') and 'safety' in str(e).lower():
                raise ValueError(
                    "Content policy violation: Your prompt was rejected by the safety system. "
                    "Please modify your prompt to comply with content policies."
                ) from e
            raise e

    def download_image(self, url: str, output_path: str) -> str:
        """
        Download image from URL and save to local path.
        """
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()  # Raise an error for HTTP errors

            with open(output_path, "wb") as out_file:
                out_file.write(response.content)
            return output_path

        except Exception as e:
            raise requests.RequestException(
                f"Failed to download image from URL: {str(e)}"
            ) from e


In [ ]:
import hashlib
import time
import os
from pathlib import Path
from typing import Optional, Dict, Any
from config import CACHE_TTL_SECONDS


class CacheManager:
    """Manages caching for generated images."""

    def __init__(self):
        """Initialize the CacheManager."""
        self.cache: Dict[str, Dict[str, Any]] = {}

    def get_cache_key(
        self,
        prompt: str,
        size: str,
        quality: str
    ) -> str:
        """
        Generate cache key from generation parameters.
        """
        combined = f"{prompt}:{size}:{quality}"
        cache_hash = hashlib.md5(combined.encode()).hexdigest()
        return f"cache:{cache_hash}"

    def check_cache(
        self,
        cache_key: str
    ) -> Optional[str]:
        """
        Check if cached image exists and is not expired.
        """
        if cache_key not in self.cache:
            return None

        entry = self.cache[cache_key]
        if time.time() - entry["timestamp"] < CACHE_TTL_SECONDS:
            del self.cache[cache_key]
            return None

        image_path = entry["image_path"] if os.path.exists(entry["image_path"]) else None

        return image_path

    def store_image_path(
        self,
        cache_key: str,
        image_path: str
    ):
        self.cache[cache_key] = {
            "image_path": image_path,
            "timestamp": time.time()
        }

    def get_image_path(
        self,
        cache_key: str
    ) -> Optional[str]:
        """
        Get cached image path.
        """
        if cache_key in self.cache:
            entry = self.cache[cache_key]

            if time.time() - entry["timestamp"] > CACHE_TTL_SECONDS:
                del self.cache[cache_key]
                return None

            return entry.get("image_path")

        return None
